In [ ]:
"""
The script is designed for a Colab/A100 run but includes small smoke test modes.
It writes CSV summaries, LaTeX tables with best values bolded, figures and a
zip ready result directory.
"""
from __future__ import annotations

import argparse
import json
import math
import os
import random
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, MutableMapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
from torch import Tensor, nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from scipy import stats as scipy_stats
except Exception:  # pragma: no cover
    scipy_stats = None


# -----------------------------------------------------------------------------
# General utilities
# -----------------------------------------------------------------------------


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except Exception:
        pass


def ensure_dirs(outdir: Path) -> Dict[str, Path]:
    paths = {
        "root": outdir,
        "figures": outdir / "figures",
        "tables": outdir / "tables",
        "csv": outdir / "csv",
        "models": outdir / "models",
        "logs": outdir / "logs",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def parse_seed_string(text: str) -> List[int]:
    """Parse seed strings such as '0-19' or '0,1,2'."""
    seeds: List[int] = []
    for part in text.replace(" ", "").split(","):
        if not part:
            continue
        if "-" in part:
            a, b = part.split("-", 1)
            seeds.extend(range(int(a), int(b) + 1))
        else:
            seeds.append(int(part))
    return sorted(dict.fromkeys(seeds))


def to_device_batch(batch: Mapping[str, Tensor], device: torch.device) -> Dict[str, Tensor]:
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def maybe_amp(enabled: bool):
    if enabled and torch.cuda.is_available():
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return torch.autocast(device_type="cpu", enabled=False)


# -----------------------------------------------------------------------------
# 2D data generation
# -----------------------------------------------------------------------------


FIXED_TEMPLATES: Tuple[Tuple[Tuple[float, float], ...], ...] = (
    ((-1.1, -0.35), (0.1, -0.35), (0.1, -0.75), (1.0, 0.0), (0.1, 0.75), (0.1, 0.35), (-1.1, 0.35), (-0.75, 0.0)),
    ((-1.0, -1.0), (-0.2, -1.0), (-0.2, 0.2), (1.0, 0.2), (1.0, 1.0), (-1.0, 1.0)),
    ((-1.0, -0.8), (0.6, -0.8), (1.0, -0.2), (1.0, 0.9), (-0.7, 0.9), (-1.0, 0.2)),
    ((0.0, -1.2), (0.9, -0.2), (0.4, 1.0), (-0.7, 0.5), (-1.0, -0.3)),
    ((-1.0, -0.9), (-0.2, -0.9), (-0.2, -0.2), (1.0, -0.2), (0.2, 0.3), (1.0, 0.8), (-0.2, 0.8), (-0.2, 1.0), (-1.0, 1.0)),
    ((-1.1, -0.6), (0.2, -0.9), (1.0, -0.2), (0.6, 0.9), (-0.7, 1.0), (-1.0, 0.2)),
)


def rasterize_polygon(points: Sequence[Tuple[float, float]], image_size: int, antialias: int = 8) -> np.ndarray:
    high_res = image_size * antialias
    im = Image.new("L", (high_res, high_res), 0)
    draw = ImageDraw.Draw(im)
    cx = cy = high_res / 2.0
    scale = 0.22 * high_res
    xy = [(cx + scale * x, cy + scale * y) for x, y in points]
    draw.polygon(xy, fill=255)
    im = im.resize((image_size, image_size), resample=Image.Resampling.BILINEAR)
    arr = np.asarray(im, dtype=np.float32) / 255.0
    return (arr > 0.5).astype(np.float32)


def make_fixed_shape_2d(image_size: int, rng: np.random.Generator, antialias: int = 8) -> np.ndarray:
    pts = FIXED_TEMPLATES[int(rng.integers(0, len(FIXED_TEMPLATES)))]
    return rasterize_polygon(pts, image_size, antialias)


def make_procedural_shape_2d(image_size: int, rng: np.random.Generator, antialias: int = 8) -> np.ndarray:
    """Random asymmetric star-like polygon used for robustness."""
    n = int(rng.integers(6, 11))
    angles = np.sort(rng.uniform(-math.pi, math.pi, size=n))
    # Make one side deliberately longer to remove accidental rotational symmetries.
    radii = rng.uniform(0.55, 1.1, size=n)
    radii[0] *= 1.30
    radii[n // 2] *= 0.70
    pts = [(float(r * math.cos(a)), float(r * math.sin(a))) for r, a in zip(radii, angles)]
    # Add a small notch/arm by perturbing two adjacent vertices.
    j = int(rng.integers(0, n))
    pts[j] = (pts[j][0] + float(rng.uniform(0.15, 0.35)), pts[j][1] - float(rng.uniform(0.05, 0.25)))
    return rasterize_polygon(pts, image_size, antialias)


def rotate_binary_image(arr: np.ndarray, angle_rad: float) -> np.ndarray:
    size = arr.shape[0]
    pil = Image.fromarray(np.uint8(arr * 255.0), mode="L")
    out = pil.rotate(
        angle=float(angle_rad * 180.0 / math.pi),
        resample=Image.Resampling.BILINEAR,
        expand=False,
        center=(size / 2.0, size / 2.0),
        fillcolor=0,
    )
    rotated = np.asarray(out, dtype=np.float32) / 255.0
    return (rotated > 0.5).astype(np.float32)


def draw_angle(rng: np.random.Generator, mode: str) -> float:
    if mode == "uniform":
        return float(rng.uniform(-math.pi, math.pi))
    if mode == "sparse45":
        grid = np.arange(-math.pi, math.pi, math.pi / 4.0, dtype=np.float64)
        return float(grid[int(rng.integers(0, len(grid)))])
    if mode == "range60":
        return float(rng.uniform(-math.pi / 3.0, math.pi / 3.0))
    raise ValueError(f"Unknown angle mode: {mode}")


class ControlledPairs2D(Dataset):
    def __init__(
        self,
        n_samples: int,
        image_size: int = 32,
        seed: int = 0,
        shape_kind: str = "fixed",
        angle_mode: str = "uniform",
        antialias: int = 8,
    ) -> None:
        rng = np.random.default_rng(seed)
        self.samples: List[Tuple[np.ndarray, np.ndarray, np.ndarray, float, float, float]] = []
        shape_fn = make_fixed_shape_2d if shape_kind == "fixed" else make_procedural_shape_2d
        for _ in range(n_samples):
            canonical = shape_fn(image_size, rng, antialias)
            theta_a = draw_angle(rng, angle_mode)
            theta_b = draw_angle(rng, angle_mode)
            x_a = rotate_binary_image(canonical, theta_a)
            x_b = rotate_binary_image(canonical, theta_b)
            self.samples.append((canonical, x_a, x_b, theta_a, theta_b, theta_b - theta_a))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Tensor]:
        c, xa, xb, ta, tb, delta = self.samples[idx]
        return {
            "canonical": torch.from_numpy(c[None]).float(),
            "x_a": torch.from_numpy(xa[None]).float(),
            "x_b": torch.from_numpy(xb[None]).float(),
            "theta_a": torch.tensor(ta, dtype=torch.float32),
            "theta_b": torch.tensor(tb, dtype=torch.float32),
            "delta": torch.tensor(delta, dtype=torch.float32),
        }


# -----------------------------------------------------------------------------
# 2D warps and models
# -----------------------------------------------------------------------------


def pose_code_from_angle(theta: Tensor) -> Tensor:
    return torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)


def rotor_from_angle(theta: Tensor) -> Tensor:
    half = 0.5 * theta
    return torch.stack([torch.cos(half), torch.sin(half)], dim=-1)


def rotor_to_active_matrix_2d(rotor: Tensor) -> Tensor:
    c, s = rotor.unbind(dim=-1)
    cos_t = c * c - s * s
    sin_t = 2.0 * c * s
    B = rotor.shape[0]
    M = torch.zeros(B, 2, 2, dtype=rotor.dtype, device=rotor.device)
    M[:, 0, 0] = cos_t
    M[:, 0, 1] = -sin_t
    M[:, 1, 0] = sin_t
    M[:, 1, 1] = cos_t
    return M


def angle_to_active_matrix_2d(theta: Tensor) -> Tensor:
    c, s = torch.cos(theta), torch.sin(theta)
    B = theta.shape[0]
    M = torch.zeros(B, 2, 2, dtype=theta.dtype, device=theta.device)
    M[:, 0, 0] = c
    M[:, 0, 1] = -s
    M[:, 1, 0] = s
    M[:, 1, 1] = c
    return M


def cos_sin_to_matrix_2d(v: Tensor, eps: float = 1e-6) -> Tensor:
    v = v / (v.norm(dim=-1, keepdim=True) + eps)
    c, s = v[:, 0], v[:, 1]
    B = v.shape[0]
    M = torch.zeros(B, 2, 2, dtype=v.dtype, device=v.device)
    M[:, 0, 0] = c
    M[:, 0, 1] = -s
    M[:, 1, 0] = s
    M[:, 1, 1] = c
    return M


def render_logits_matrix_2d(logits: Tensor, R: Tensor) -> Tensor:
    B = logits.shape[0]
    A = torch.zeros(B, 2, 3, dtype=logits.dtype, device=logits.device)
    A[:, :, :2] = R.transpose(1, 2)
    grid = F.affine_grid(A, logits.size(), align_corners=False)
    return F.grid_sample(logits, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


def render_logits_rotor_2d(logits: Tensor, rotor: Tensor) -> Tensor:
    return render_logits_matrix_2d(logits, rotor_to_active_matrix_2d(rotor))


class Encoder2D(nn.Module):
    def __init__(self, z_dim: int, image_size: int = 32, base_channels: int = 8):
        super().__init__()
        if image_size % 16 != 0:
            raise ValueError("image_size must be divisible by 16")
        c1, c2, c3, c4 = base_channels, 2 * base_channels, 4 * base_channels, 8 * base_channels
        hw = image_size // 16
        self.net = nn.Sequential(
            nn.Conv2d(1, c1, 4, 2, 1), nn.SiLU(),
            nn.Conv2d(c1, c2, 4, 2, 1), nn.SiLU(),
            nn.Conv2d(c2, c3, 4, 2, 1), nn.SiLU(),
            nn.Conv2d(c3, c4, 4, 2, 1), nn.SiLU(),
            nn.Flatten(),
            nn.Linear(c4 * hw * hw, 256), nn.SiLU(),
        )
        self.mu = nn.Linear(256, z_dim)
        self.logvar = nn.Linear(256, z_dim)

    def forward(self, x: Tensor) -> Dict[str, Tensor]:
        h = self.net(x)
        return {"mu_z": self.mu(h), "logvar_z": self.logvar(h)}


class Decoder2D(nn.Module):
    def __init__(self, input_dim: int, image_size: int = 32, base_channels: int = 8):
        super().__init__()
        c1, c2, c3, c4 = base_channels, 2 * base_channels, 4 * base_channels, 8 * base_channels
        hw = image_size // 16
        self.hw, self.c4 = hw, c4
        self.fc = nn.Sequential(nn.Linear(input_dim, 256), nn.SiLU(), nn.Linear(256, c4 * hw * hw), nn.SiLU())
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(c4, c3, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(c3, c2, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(c2, c1, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(c1, 1, 4, 2, 1),
        )

    def forward(self, z: Tensor) -> Tensor:
        h = self.fc(z).view(-1, self.c4, self.hw, self.hw)
        return self.deconv(h)


def reparameterize(mu: Tensor, logvar: Tensor) -> Tensor:
    std = torch.exp(0.5 * logvar)
    return mu + torch.randn_like(std) * std


def shared_latent(enc_a: Mapping[str, Tensor], enc_b: Mapping[str, Tensor]) -> Tuple[Tensor, Tensor]:
    za = reparameterize(enc_a["mu_z"], enc_a["logvar_z"])
    zb = reparameterize(enc_b["mu_z"], enc_b["logvar_z"])
    return 0.5 * (za + zb), 0.5 * (enc_a["mu_z"] + enc_b["mu_z"])


class AnalyticWarpVAE2D(nn.Module):
    """Analytic warp pathway. mode='rotor' or mode='matrix'."""
    def __init__(self, z_dim: int = 16, image_size: int = 32, base_channels: int = 8, mode: str = "rotor"):
        super().__init__()
        self.encoder = Encoder2D(z_dim, image_size, base_channels)
        self.decoder = Decoder2D(z_dim, image_size, base_channels)
        self.mode = mode

    def _R(self, theta: Tensor) -> Tensor:
        # Positive raster angle corresponds to active Euclidean angle -theta.
        if self.mode == "rotor":
            return rotor_to_active_matrix_2d(rotor_from_angle(-theta))
        if self.mode == "matrix":
            return angle_to_active_matrix_2d(-theta)
        raise ValueError(self.mode)

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a = self.encoder(batch["x_a"]); enc_b = self.encoder(batch["x_b"])
        z_s, mu_s = shared_latent(enc_a, enc_b)
        can = self.decoder(z_s); can_mean = self.decoder(mu_s)
        R_a, R_b = self._R(batch["theta_a"]), self._R(batch["theta_b"])
        return {
            "enc_a": enc_a, "enc_b": enc_b,
            "canonical_logits": can, "canonical_logits_mean": can_mean,
            "recon_a": render_logits_matrix_2d(can, R_a),
            "recon_b": render_logits_matrix_2d(can, R_b),
            "recon_a_mean": render_logits_matrix_2d(can_mean, R_a),
            "recon_b_mean": render_logits_matrix_2d(can_mean, R_b),
        }


class CoordinateDecoderVAE2D(nn.Module):
    def __init__(self, z_dim: int = 16, image_size: int = 32, base_channels: int = 8):
        super().__init__()
        self.encoder = Encoder2D(z_dim, image_size, base_channels)
        self.decoder = Decoder2D(z_dim + 2, image_size, base_channels)

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a = self.encoder(batch["x_a"]); enc_b = self.encoder(batch["x_b"])
        z_s, mu_s = shared_latent(enc_a, enc_b)
        p0 = torch.zeros(z_s.shape[0], 2, dtype=z_s.dtype, device=z_s.device); p0[:, 0] = 1.0
        pa, pb = pose_code_from_angle(batch["theta_a"]), pose_code_from_angle(batch["theta_b"])
        return {
            "enc_a": enc_a, "enc_b": enc_b,
            "canonical_logits": self.decoder(torch.cat([z_s, p0], -1)),
            "canonical_logits_mean": self.decoder(torch.cat([mu_s, p0], -1)),
            "recon_a": self.decoder(torch.cat([z_s, pa], -1)),
            "recon_b": self.decoder(torch.cat([z_s, pb], -1)),
            "recon_a_mean": self.decoder(torch.cat([mu_s, pa], -1)),
            "recon_b_mean": self.decoder(torch.cat([mu_s, pb], -1)),
        }


class LearnedWarpVAE2D(nn.Module):
    def __init__(self, z_dim: int = 16, image_size: int = 32, base_channels: int = 8):
        super().__init__()
        self.encoder = Encoder2D(z_dim, image_size, base_channels)
        self.decoder = Decoder2D(z_dim, image_size, base_channels)
        self.pose_mlp = nn.Sequential(nn.Linear(2, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))

    def _R_from_pose(self, theta: Tensor) -> Tensor:
        p = pose_code_from_angle(theta)
        v = self.pose_mlp(p)
        return cos_sin_to_matrix_2d(v)

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a = self.encoder(batch["x_a"]); enc_b = self.encoder(batch["x_b"])
        z_s, mu_s = shared_latent(enc_a, enc_b)
        can = self.decoder(z_s); can_mean = self.decoder(mu_s)
        R_a, R_b = self._R_from_pose(batch["theta_a"]), self._R_from_pose(batch["theta_b"])
        return {
            "enc_a": enc_a, "enc_b": enc_b,
            "canonical_logits": can, "canonical_logits_mean": can_mean,
            "recon_a": render_logits_matrix_2d(can, R_a),
            "recon_b": render_logits_matrix_2d(can, R_b),
            "recon_a_mean": render_logits_matrix_2d(can_mean, R_a),
            "recon_b_mean": render_logits_matrix_2d(can_mean, R_b),
        }


# -----------------------------------------------------------------------------
# Losses, metrics, 2D training
# -----------------------------------------------------------------------------


@dataclass(frozen=True)
class LossWeights:
    beta_z: float = 1e-4
    lambda_can_bce: float = 5.0
    lambda_can_dice: float = 2.0
    lambda_view_dice: float = 1.0
    lambda_inv: float = 5.0
    lambda_conf: float = 0.02


LOSS_VARIANTS: Dict[str, LossWeights] = {
    "full": LossWeights(),
    "no_conf": replace(LossWeights(), lambda_conf=0.0),
    "no_inv": replace(LossWeights(), lambda_inv=0.0),
    "no_can_dice": replace(LossWeights(), lambda_can_dice=0.0),
    "no_view_dice": replace(LossWeights(), lambda_view_dice=0.0),
    "no_can_supervision": replace(LossWeights(), lambda_can_bce=0.0, lambda_can_dice=0.0),
    "bce_only": LossWeights(lambda_can_bce=1.0, lambda_can_dice=0.0, lambda_view_dice=0.0, lambda_inv=0.0, lambda_conf=0.0, beta_z=1e-4),
}


def kl_standard_normal(mu: Tensor, logvar: Tensor) -> Tensor:
    return 0.5 * torch.sum(torch.exp(logvar) + mu * mu - 1.0 - logvar, dim=-1)


def soft_dice_coeff(logits: Tensor, target: Tensor, eps: float = 1e-6) -> Tensor:
    p = torch.sigmoid(logits)
    inter = (p * target).flatten(1).sum(1)
    denom = p.flatten(1).sum(1) + target.flatten(1).sum(1)
    return (2 * inter + eps) / (denom + eps)


def hard_dice_coeff(logits: Tensor, target: Tensor, eps: float = 1e-6) -> Tensor:
    p = (torch.sigmoid(logits) > 0.5).float()
    inter = (p * target).flatten(1).sum(1)
    denom = p.flatten(1).sum(1) + target.flatten(1).sum(1)
    return (2 * inter + eps) / (denom + eps)


def confidence_penalty(logits: Tensor) -> Tensor:
    p = torch.sigmoid(logits)
    return (p * (1 - p)).mean()


def binary_entropy(logits: Tensor, eps: float = 1e-6) -> Tensor:
    p = torch.sigmoid(logits).clamp(eps, 1 - eps)
    return (-(p * torch.log(p) + (1 - p) * torch.log(1 - p))).mean()


def compute_common_losses(batch: Mapping[str, Tensor], outputs: Mapping[str, Tensor], weights: LossWeights) -> Dict[str, Tensor]:
    xa, xb, c = batch["x_a"], batch["x_b"], batch["canonical"]
    ra, rb = outputs["recon_a"], outputs["recon_b"]
    ram, rbm = outputs["recon_a_mean"], outputs["recon_b_mean"]
    canm = outputs["canonical_logits_mean"]
    enc_a, enc_b = outputs["enc_a"], outputs["enc_b"]
    view_bce = F.binary_cross_entropy_with_logits(ra, xa) + F.binary_cross_entropy_with_logits(rb, xb)
    view_dice_loss = (1 - soft_dice_coeff(ra, xa)).mean() + (1 - soft_dice_coeff(rb, xb)).mean()
    can_bce = F.binary_cross_entropy_with_logits(canm, c)
    can_dice_loss = (1 - soft_dice_coeff(canm, c)).mean()
    inv = F.mse_loss(enc_a["mu_z"], enc_b["mu_z"])
    kl = kl_standard_normal(enc_a["mu_z"], enc_a["logvar_z"]).mean() + kl_standard_normal(enc_b["mu_z"], enc_b["logvar_z"]).mean()
    conf = confidence_penalty(canm) + confidence_penalty(ram) + confidence_penalty(rbm)
    loss = view_bce + weights.lambda_view_dice * view_dice_loss + weights.lambda_can_bce * can_bce + weights.lambda_can_dice * can_dice_loss + weights.lambda_inv * inv + weights.beta_z * kl + weights.lambda_conf * conf
    return {
        "loss": loss,
        "view_bce": view_bce,
        "canonical_bce": can_bce,
        "canonical_hard_dice": hard_dice_coeff(canm, c).mean(),
        "view_hard_dice": 0.5 * (hard_dice_coeff(ram, xa).mean() + hard_dice_coeff(rbm, xb).mean()),
        "invariance": inv,
        "kl_z": kl,
        "confidence": conf,
        "entropy": (binary_entropy(canm) + binary_entropy(ram) + binary_entropy(rbm)) / 3.0,
    }


@torch.no_grad()
def composition_consistency_mse_2d(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    vals: List[float] = []
    for batch in loader:
        batch = to_device_batch(batch, device)
        outputs = model.forward_pair(batch)
        R_rel = angle_to_active_matrix_2d(-batch["delta"])
        b_from_a = render_logits_matrix_2d(outputs["recon_a_mean"], R_rel)
        vals.append(float(F.mse_loss(torch.sigmoid(b_from_a), torch.sigmoid(outputs["recon_b_mean"])).detach().cpu()))
    return float(np.mean(vals))


@dataclass(frozen=True)
class Config2D:
    image_size: int = 32
    num_train: int = 256
    num_val: int = 128
    batch_size: int = 32
    epochs: int = 32
    z_dim: int = 16
    base_channels: int = 8
    learning_rate: float = 2e-3
    shape_kind: str = "fixed"
    train_angle_mode: str = "uniform"
    val_angle_mode: str = "uniform"
    num_workers: int = 0
    amp: bool = False


def build_2d_model(model_name: str, cfg: Config2D) -> nn.Module:
    if model_name == "coord":
        return CoordinateDecoderVAE2D(cfg.z_dim, cfg.image_size, cfg.base_channels)
    if model_name == "learned_warp":
        return LearnedWarpVAE2D(cfg.z_dim, cfg.image_size, cfg.base_channels)
    if model_name == "so2_stn":
        return AnalyticWarpVAE2D(cfg.z_dim, cfg.image_size, cfg.base_channels, mode="matrix")
    if model_name == "ga_rotor":
        return AnalyticWarpVAE2D(cfg.z_dim, cfg.image_size, cfg.base_channels, mode="rotor")
    raise ValueError(model_name)


def build_loaders_2d(cfg: Config2D, seed: int) -> Tuple[DataLoader, DataLoader, Dict[str, Tensor]]:
    train_ds = ControlledPairs2D(cfg.num_train, cfg.image_size, seed=seed, shape_kind=cfg.shape_kind, angle_mode=cfg.train_angle_mode)
    val_ds = ControlledPairs2D(cfg.num_val, cfg.image_size, seed=seed + 100_000, shape_kind=cfg.shape_kind, angle_mode=cfg.val_angle_mode)
    gen = torch.Generator(); gen.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, generator=gen, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader, next(iter(val_loader))


def run_epoch_2d(model: nn.Module, loader: DataLoader, device: torch.device, weights: LossWeights, optimizer: Optional[torch.optim.Optimizer], amp: bool = False) -> Dict[str, float]:
    training = optimizer is not None
    model.train(training)
    sums: Dict[str, float] = {}
    count = 0
    for batch in loader:
        batch = to_device_batch(batch, device)
        with maybe_amp(amp):
            outputs = model.forward_pair(batch)
            losses = compute_common_losses(batch, outputs, weights)
        if training:
            optimizer.zero_grad(set_to_none=True)
            losses["loss"].backward()
            optimizer.step()
        n = int(batch["x_a"].shape[0])
        count += n
        for k, v in losses.items():
            sums[k] = sums.get(k, 0.0) + float(v.detach().cpu()) * n
    return {k: v / max(1, count) for k, v in sums.items()}


def train_one_2d(model_name: str, cfg: Config2D, seed: int, device: torch.device, weights: LossWeights, save_model_path: Optional[Path] = None) -> Tuple[Dict[str, float], Optional[nn.Module], Optional[Dict[str, Tensor]]]:
    set_seed(seed)
    train_loader, val_loader, rep_batch = build_loaders_2d(cfg, seed)
    model = build_2d_model(model_name, cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    last_val: Dict[str, float] = {}
    for epoch in range(1, cfg.epochs + 1):
        _ = run_epoch_2d(model, train_loader, device, weights, opt, amp=cfg.amp)
        last_val = run_epoch_2d(model, val_loader, device, weights, None, amp=False)
    last_val["composition_mse"] = composition_consistency_mse_2d(model, val_loader, device)
    if save_model_path is not None:
        torch.save(model.state_dict(), save_model_path)
        return last_val, model, rep_batch
    return last_val, None, None


# -----------------------------------------------------------------------------
# 3D data, models and training
# -----------------------------------------------------------------------------


def random_quaternion(rng: np.random.Generator) -> np.ndarray:
    # Shoemake method, returns [w,x,y,z]
    u1, u2, u3 = rng.random(3)
    q = np.array([
        math.sqrt(1 - u1) * math.sin(2 * math.pi * u2),
        math.sqrt(1 - u1) * math.cos(2 * math.pi * u2),
        math.sqrt(u1) * math.sin(2 * math.pi * u3),
        math.sqrt(u1) * math.cos(2 * math.pi * u3),
    ], dtype=np.float32)
    # Convert [x,y,z,w] to [w,x,y,z]
    return np.array([q[3], q[0], q[1], q[2]], dtype=np.float32)


def quat_to_matrix_np(q: np.ndarray) -> np.ndarray:
    w, x, y, z = [float(v) for v in q]
    n = math.sqrt(w*w + x*x + y*y + z*z) + 1e-12
    w, x, y, z = w/n, x/n, y/n, z/n
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*z*w, 2*x*z + 2*y*w],
        [2*x*y + 2*z*w, 1 - 2*x*x - 2*z*z, 2*y*z - 2*x*w],
        [2*x*z - 2*y*w, 2*y*z + 2*x*w, 1 - 2*x*x - 2*y*y],
    ], dtype=np.float32)


def rotation_6d_from_matrix_np(R: np.ndarray) -> np.ndarray:
    # Zhou et al. 6D representation: concatenate the first two rotation columns.
    return np.concatenate([R[:, 0], R[:, 1]], axis=0).astype(np.float32)


def rotation_6d_to_matrix_torch(x: Tensor, eps: float = 1e-6) -> Tensor:
    a1 = x[:, 0:3]
    a2 = x[:, 3:6]
    b1 = F.normalize(a1, dim=1, eps=eps)
    b2 = F.normalize(a2 - (b1 * a2).sum(dim=1, keepdim=True) * b1, dim=1, eps=eps)
    b3 = torch.cross(b1, b2, dim=1)
    return torch.stack([b1, b2, b3], dim=2)  # columns


def make_random_voxel_shape(size: int, rng: np.random.Generator) -> np.ndarray:
    lin = np.linspace(-1, 1, size, dtype=np.float32)
    z, y, x = np.meshgrid(lin, lin, lin, indexing="ij")
    vol = np.zeros((size, size, size), dtype=bool)
    num_prims = int(rng.integers(3, 6))
    for i in range(num_prims):
        center = rng.uniform(-0.35, 0.35, size=3)
        radii = rng.uniform(0.18, 0.48, size=3)
        if i == 0:
            center += np.array([0.20, -0.12, 0.08])
            radii *= np.array([1.30, 0.75, 0.90])
        if rng.random() < 0.55:
            mask = ((x-center[0])/radii[0])**2 + ((y-center[1])/radii[1])**2 + ((z-center[2])/radii[2])**2 <= 1.0
        else:
            mask = (np.abs(x-center[0]) <= radii[0]) & (np.abs(y-center[1]) <= radii[1]) & (np.abs(z-center[2]) <= radii[2])
        vol |= mask
    # Add an asymmetric arm.
    arm = (x > 0.05) & (x < 0.75) & (y > -0.15) & (y < 0.05) & (z > 0.05) & (z < 0.32)
    vol |= arm
    return vol.astype(np.float32)


def render_logits_matrix_3d(logits: Tensor, R: Tensor) -> Tensor:
    B = logits.shape[0]
    A = torch.zeros(B, 3, 4, dtype=logits.dtype, device=logits.device)
    A[:, :, :3] = R.transpose(1, 2)
    grid = F.affine_grid(A, logits.size(), align_corners=False)
    return F.grid_sample(logits, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


def rotate_volume_np(vol: np.ndarray, R: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        x = torch.from_numpy(vol[None, None].astype(np.float32))
        Rt = torch.from_numpy(R[None].astype(np.float32))
        out = render_logits_matrix_3d(x, Rt)[0, 0].cpu().numpy()
    return (out > 0.5).astype(np.float32)


class ControlledPairs3D(Dataset):
    def __init__(self, n_samples: int, size: int = 32, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.samples: List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = []
        for _ in range(n_samples):
            can = make_random_voxel_shape(size, rng)
            qa, qb = random_quaternion(rng), random_quaternion(rng)
            Ra, Rb = quat_to_matrix_np(qa), quat_to_matrix_np(qb)
            xa = rotate_volume_np(can, Ra)
            xb = rotate_volume_np(can, Rb)
            self.samples.append((can, xa, xb, Ra, Rb, rotation_6d_from_matrix_np(Ra), rotation_6d_from_matrix_np(Rb)))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Tensor]:
        can, xa, xb, Ra, Rb, p6a, p6b = self.samples[idx]
        return {
            "canonical": torch.from_numpy(can[None]).float(),
            "x_a": torch.from_numpy(xa[None]).float(),
            "x_b": torch.from_numpy(xb[None]).float(),
            "R_a": torch.from_numpy(Ra).float(),
            "R_b": torch.from_numpy(Rb).float(),
            "pose6_a": torch.from_numpy(p6a).float(),
            "pose6_b": torch.from_numpy(p6b).float(),
        }


class Encoder3D(nn.Module):
    def __init__(self, z_dim: int, size: int = 32, base_channels: int = 8):
        super().__init__()
        c1, c2, c3, c4 = base_channels, 2*base_channels, 4*base_channels, 8*base_channels
        hw = size // 16
        self.net = nn.Sequential(
            nn.Conv3d(1, c1, 4, 2, 1), nn.SiLU(),
            nn.Conv3d(c1, c2, 4, 2, 1), nn.SiLU(),
            nn.Conv3d(c2, c3, 4, 2, 1), nn.SiLU(),
            nn.Conv3d(c3, c4, 4, 2, 1), nn.SiLU(),
            nn.Flatten(),
            nn.Linear(c4 * hw * hw * hw, 256), nn.SiLU(),
        )
        self.mu = nn.Linear(256, z_dim)
        self.logvar = nn.Linear(256, z_dim)

    def forward(self, x: Tensor) -> Dict[str, Tensor]:
        h = self.net(x)
        return {"mu_z": self.mu(h), "logvar_z": self.logvar(h)}


class Decoder3D(nn.Module):
    def __init__(self, input_dim: int, size: int = 32, base_channels: int = 8):
        super().__init__()
        c1, c2, c3, c4 = base_channels, 2*base_channels, 4*base_channels, 8*base_channels
        hw = size // 16
        self.hw, self.c4 = hw, c4
        self.fc = nn.Sequential(nn.Linear(input_dim, 256), nn.SiLU(), nn.Linear(256, c4 * hw * hw * hw), nn.SiLU())
        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(c4, c3, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose3d(c3, c2, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose3d(c2, c1, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose3d(c1, 1, 4, 2, 1),
        )

    def forward(self, z: Tensor) -> Tensor:
        h = self.fc(z).view(-1, self.c4, self.hw, self.hw, self.hw)
        return self.deconv(h)


class AnalyticWarpVAE3D(nn.Module):
    def __init__(self, z_dim: int = 32, size: int = 32, base_channels: int = 8):
        super().__init__()
        self.encoder = Encoder3D(z_dim, size, base_channels)
        self.decoder = Decoder3D(z_dim, size, base_channels)

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a = self.encoder(batch["x_a"]); enc_b = self.encoder(batch["x_b"])
        z_s, mu_s = shared_latent(enc_a, enc_b)
        can = self.decoder(z_s); canm = self.decoder(mu_s)
        return {
            "enc_a": enc_a, "enc_b": enc_b,
            "canonical_logits": can, "canonical_logits_mean": canm,
            "recon_a": render_logits_matrix_3d(can, batch["R_a"]),
            "recon_b": render_logits_matrix_3d(can, batch["R_b"]),
            "recon_a_mean": render_logits_matrix_3d(canm, batch["R_a"]),
            "recon_b_mean": render_logits_matrix_3d(canm, batch["R_b"]),
        }


class CoordinateDecoderVAE3D(nn.Module):
    def __init__(self, z_dim: int = 32, size: int = 32, base_channels: int = 8):
        super().__init__()
        self.encoder = Encoder3D(z_dim, size, base_channels)
        self.decoder = Decoder3D(z_dim + 6, size, base_channels)
        self.register_buffer("identity6", torch.tensor([1,0,0,0,1,0], dtype=torch.float32))

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a = self.encoder(batch["x_a"]); enc_b = self.encoder(batch["x_b"])
        z_s, mu_s = shared_latent(enc_a, enc_b)
        p0 = self.identity6[None].expand(z_s.shape[0], -1).to(z_s)
        return {
            "enc_a": enc_a, "enc_b": enc_b,
            "canonical_logits": self.decoder(torch.cat([z_s, p0], -1)),
            "canonical_logits_mean": self.decoder(torch.cat([mu_s, p0], -1)),
            "recon_a": self.decoder(torch.cat([z_s, batch["pose6_a"]], -1)),
            "recon_b": self.decoder(torch.cat([z_s, batch["pose6_b"]], -1)),
            "recon_a_mean": self.decoder(torch.cat([mu_s, batch["pose6_a"]], -1)),
            "recon_b_mean": self.decoder(torch.cat([mu_s, batch["pose6_b"]], -1)),
        }


class LearnedWarpVAE3D(nn.Module):
    def __init__(self, z_dim: int = 32, size: int = 32, base_channels: int = 8):
        super().__init__()
        self.encoder = Encoder3D(z_dim, size, base_channels)
        self.decoder = Decoder3D(z_dim, size, base_channels)
        self.pose_mlp = nn.Sequential(nn.Linear(6, 96), nn.SiLU(), nn.Linear(96, 96), nn.SiLU(), nn.Linear(96, 6))

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a = self.encoder(batch["x_a"]); enc_b = self.encoder(batch["x_b"])
        z_s, mu_s = shared_latent(enc_a, enc_b)
        can = self.decoder(z_s); canm = self.decoder(mu_s)
        Ra = rotation_6d_to_matrix_torch(self.pose_mlp(batch["pose6_a"]))
        Rb = rotation_6d_to_matrix_torch(self.pose_mlp(batch["pose6_b"]))
        return {
            "enc_a": enc_a, "enc_b": enc_b,
            "canonical_logits": can, "canonical_logits_mean": canm,
            "recon_a": render_logits_matrix_3d(can, Ra),
            "recon_b": render_logits_matrix_3d(can, Rb),
            "recon_a_mean": render_logits_matrix_3d(canm, Ra),
            "recon_b_mean": render_logits_matrix_3d(canm, Rb),
        }


@dataclass(frozen=True)
class Config3D:
    size: int = 32
    num_train: int = 512
    num_val: int = 128
    batch_size: int = 8
    epochs: int = 24
    z_dim: int = 32
    base_channels: int = 8
    learning_rate: float = 1e-3
    num_workers: int = 0
    amp: bool = False


def build_3d_model(name: str, cfg: Config3D) -> nn.Module:
    if name == "coord3d":
        return CoordinateDecoderVAE3D(cfg.z_dim, cfg.size, cfg.base_channels)
    if name == "learned_warp3d":
        return LearnedWarpVAE3D(cfg.z_dim, cfg.size, cfg.base_channels)
    if name in {"so3_matrix", "rotor3d"}:
        return AnalyticWarpVAE3D(cfg.z_dim, cfg.size, cfg.base_channels)
    raise ValueError(name)


def build_loaders_3d(cfg: Config3D, seed: int) -> Tuple[DataLoader, DataLoader, Dict[str, Tensor]]:
    train_ds = ControlledPairs3D(cfg.num_train, cfg.size, seed=seed)
    val_ds = ControlledPairs3D(cfg.num_val, cfg.size, seed=seed + 100_000)
    gen = torch.Generator(); gen.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, generator=gen, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader, next(iter(val_loader))


@torch.no_grad()
def composition_consistency_mse_3d(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    vals: List[float] = []
    for batch in loader:
        batch = to_device_batch(batch, device)
        outputs = model.forward_pair(batch)
        R_rel = torch.bmm(batch["R_b"], batch["R_a"].transpose(1, 2))
        b_from_a = render_logits_matrix_3d(outputs["recon_a_mean"], R_rel)
        vals.append(float(F.mse_loss(torch.sigmoid(b_from_a), torch.sigmoid(outputs["recon_b_mean"])).detach().cpu()))
    return float(np.mean(vals))


def run_epoch_3d(model: nn.Module, loader: DataLoader, device: torch.device, weights: LossWeights, optimizer: Optional[torch.optim.Optimizer], amp: bool = False) -> Dict[str, float]:
    training = optimizer is not None
    model.train(training)
    sums: Dict[str, float] = {}
    count = 0
    for batch in loader:
        batch = to_device_batch(batch, device)
        with maybe_amp(amp):
            outputs = model.forward_pair(batch)
            losses = compute_common_losses(batch, outputs, weights)
        if training:
            optimizer.zero_grad(set_to_none=True)
            losses["loss"].backward()
            optimizer.step()
        n = int(batch["x_a"].shape[0])
        count += n
        for k, v in losses.items():
            sums[k] = sums.get(k, 0.0) + float(v.detach().cpu()) * n
    return {k: v / max(1, count) for k, v in sums.items()}


def train_one_3d(model_name: str, cfg: Config3D, seed: int, device: torch.device, weights: LossWeights, save_model_path: Optional[Path] = None) -> Tuple[Dict[str, float], Optional[nn.Module], Optional[Dict[str, Tensor]]]:
    set_seed(seed)
    train_loader, val_loader, rep_batch = build_loaders_3d(cfg, seed)
    model = build_3d_model(model_name, cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    last_val: Dict[str, float] = {}
    for _ in range(cfg.epochs):
        _ = run_epoch_3d(model, train_loader, device, weights, opt, amp=cfg.amp)
        last_val = run_epoch_3d(model, val_loader, device, weights, None, amp=False)
    last_val["composition_mse"] = composition_consistency_mse_3d(model, val_loader, device)
    if save_model_path is not None:
        torch.save(model.state_dict(), save_model_path)
        return last_val, model, rep_batch
    return last_val, None, None


# -----------------------------------------------------------------------------
# Summaries, tables and figures
# -----------------------------------------------------------------------------


METRICS: Tuple[Tuple[str, str, str], ...] = (
    ("canonical_bce", "Canonical BCE", "lower"),
    ("canonical_hard_dice", "Canonical Dice", "higher"),
    ("view_bce", "View BCE", "lower"),
    ("view_hard_dice", "View Dice", "higher"),
    ("composition_mse", "Composition MSE", "lower"),
)

MODEL_LABELS = {
    "coord": "Coord. decoder",
    "learned_warp": "Learned warp",
    "so2_stn": "Exact $\\SO(2)$/STN",
    "ga_rotor": "GA rotor",
    "coord3d": "Coord. decoder",
    "learned_warp3d": "Learned warp",
    "so3_matrix": "Exact $\\SO(3)$ matrix",
    "rotor3d": "3D rotor/quaternion",
}


def mean_std(arr: Sequence[float]) -> Tuple[float, float]:
    x = np.asarray(arr, dtype=np.float64)
    return float(x.mean()), float(x.std(ddof=1 if len(x) > 1 else 0.0))


def format_value(mean: float, std: float, bold: bool = False) -> str:
    s = f"{mean:.4f} $\\pm$ {std:.4f}"
    return f"\\best{{{s}}}" if bold else s



def write_summary_table(df: pd.DataFrame, model_order: Sequence[str], outpath: Path, metrics: Sequence[Tuple[str, str, str]] = METRICS, group_cols: Sequence[str] = ("model",)) -> None:
    """Write a compact LaTeX table with best values in bold."""
    if tuple(group_cols) != ("model",):
        raise NotImplementedError
    row_end = r" \\"
    best_models: Dict[str, str] = {}
    for key, label, direction in metrics:
        vals = []
        for m in model_order:
            sub = df[df["model"] == m][key].astype(float).values
            vals.append(np.nan if len(sub) == 0 else float(np.mean(sub)))
        best_idx = int(np.nanargmin(vals) if direction == "lower" else np.nanargmax(vals))
        best_models[key] = model_order[best_idx]

    header_cells = []
    for _, label, direction in metrics:
        arrow = r"$\downarrow$" if direction == "lower" else r"$\uparrow$"
        header_cells.append(f"{label} {arrow}")
    lines = [r"\begin{tabular}{l" + "c" * len(metrics) + "}", r"\toprule"]
    lines.append("Model & " + " & ".join(header_cells) + row_end)
    lines.append(r"\midrule")
    for m in model_order:
        cells = [MODEL_LABELS.get(m, m)]
        for key, _, _ in metrics:
            sub = df[df["model"] == m][key].astype(float).values
            if len(sub) == 0:
                cells.append("--")
            else:
                mn, sd = mean_std(sub)
                cells.append(format_value(mn, sd, bold=(m == best_models[key])))
        lines.append(" & ".join(cells) + row_end)
    lines += [r"\bottomrule", r"\end{tabular}"]
    outpath.write_text("\n".join(lines), encoding="utf-8")


def paired_tests_table(df: pd.DataFrame, baseline: str, comparisons: Sequence[str], outpath: Path, metrics: Sequence[Tuple[str, str, str]] = METRICS) -> None:
    """Write paired seed-wise tests against the coordinate baseline."""
    row_end = r" \\"
    lines = [r"\begin{tabular}{llccc}", r"\toprule", r"Metric & Comparison & Mean diff & 95\% CI & $p_t$" + row_end, r"\midrule"]
    for key, label, direction in metrics:
        base = df[df["model"] == baseline].set_index("seed")[key]
        for comp in comparisons:
            other = df[df["model"] == comp].set_index("seed")[key]
            common = sorted(set(base.index) & set(other.index))
            if not common:
                continue
            b = base.loc[common].astype(float).values
            o = other.loc[common].astype(float).values
            diff = (b - o) if direction == "lower" else (o - b)  # positive favours comp over baseline
            md = float(np.mean(diff))
            sd = float(np.std(diff, ddof=1 if len(diff) > 1 else 0))
            se = sd / math.sqrt(max(1, len(diff)))
            if scipy_stats is not None and len(diff) > 1:
                tcrit = float(scipy_stats.t.ppf(0.975, df=len(diff) - 1))
                pt = float(scipy_stats.ttest_rel(o, b).pvalue)
            else:
                tcrit, pt = 1.96, float("nan")
            lo, hi = md - tcrit * se, md + tcrit * se
            lines.append(f"{label} & {MODEL_LABELS.get(comp, comp)} vs {MODEL_LABELS.get(baseline, baseline)} & {md:.4f} & [{lo:.4f}, {hi:.4f}] & {pt:.3g}" + row_end)
    lines += [r"\bottomrule", r"\end{tabular}"]
    outpath.write_text("\n".join(lines), encoding="utf-8")

def save_metric_bar_figure(df: pd.DataFrame, model_order: Sequence[str], outpath: Path, metrics: Sequence[Tuple[str, str, str]] = METRICS) -> None:
    fig, axes = plt.subplots(1, len(metrics), figsize=(3.0 * len(metrics), 3.0))
    if len(metrics) == 1:
        axes = [axes]
    for ax, (key, label, direction) in zip(axes, metrics):
        means, stds = [], []
        for m in model_order:
            vals = df[df["model"] == m][key].astype(float).values
            mn, sd = mean_std(vals)
            means.append(mn); stds.append(sd)
        x = np.arange(len(model_order))
        ax.bar(x, means, yerr=stds, capsize=3)
        ax.set_xticks(x)
        ax.set_xticklabels([MODEL_LABELS.get(m, m).replace("$\\SO(2)$", "SO(2)").replace("$\\SO(3)$", "SO(3)") for m in model_order],rotation=45, ha="right", fontsize=8)
        ax.set_title(label)
        ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def save_2d_qualitative(model: nn.Module, batch: Mapping[str, Tensor], device: torch.device, outpath: Path, max_items: int = 4) -> None:
    model.eval()
    with torch.no_grad():
        batch_d = to_device_batch(batch, device)
        out = model.forward_pair(batch_d)
        true_can = batch["canonical"][:max_items, 0]
        true_a = batch["x_a"][:max_items, 0]
        true_b = batch["x_b"][:max_items, 0]
        pred_can = (torch.sigmoid(out["canonical_logits_mean"][:max_items, 0]).cpu() > 0.5).float()
        pred_a = (torch.sigmoid(out["recon_a_mean"][:max_items, 0]).cpu() > 0.5).float()
        pred_b = (torch.sigmoid(out["recon_b_mean"][:max_items, 0]).cpu() > 0.5).float()
    fig, axes = plt.subplots(5, max_items, figsize=(2.2 * max_items, 7.0))
    rows = [true_can, pred_can, true_a, pred_a, pred_b]
    labels = ["True canon", "Pred canon", "True A", "Pred A", "Pred B"]
    for r, (imgs, label) in enumerate(zip(rows, labels)):
        for c in range(max_items):
            axes[r, c].imshow(imgs[c], cmap="gray", vmin=0, vmax=1)
            axes[r, c].axis("off")
            if c == 0:
                axes[r, c].set_ylabel(label, fontsize=9)
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def save_3d_qualitative(model: nn.Module, batch: Mapping[str, Tensor], device: torch.device, outpath: Path) -> None:
    model.eval()
    with torch.no_grad():
        batch_d = to_device_batch(batch, device)
        out = model.forward_pair(batch_d)
        idx = 0
        can = batch["canonical"][idx, 0].numpy()
        xa = batch["x_a"][idx, 0].numpy()
        xb = batch["x_b"][idx, 0].numpy()
        pred_can = (torch.sigmoid(out["canonical_logits_mean"][idx, 0]).cpu().numpy() > 0.5).astype(np.float32)
        pred_a = (torch.sigmoid(out["recon_a_mean"][idx, 0]).cpu().numpy() > 0.5).astype(np.float32)
        pred_b = (torch.sigmoid(out["recon_b_mean"][idx, 0]).cpu().numpy() > 0.5).astype(np.float32)
    mid = can.shape[0] // 2
    vols = [can, pred_can, xa, pred_a, xb, pred_b]
    labels = ["true canon", "pred canon", "true A", "pred A", "true B", "pred B"]
    fig, axes = plt.subplots(3, len(vols), figsize=(2.0 * len(vols), 5.8))
    for col, (vol, label) in enumerate(zip(vols, labels)):
        slices = [vol[mid, :, :], vol[:, mid, :], vol[:, :, mid]]
        for row in range(3):
            axes[row, col].imshow(slices[row], cmap="gray", vmin=0, vmax=1)
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(label, fontsize=8)
            if col == 0:
                axes[row, col].set_ylabel(["z mid", "y mid", "x mid"][row], fontsize=8)
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def write_json(obj: object, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")


# -----------------------------------------------------------------------------
# Experiment drivers
# -----------------------------------------------------------------------------


def run_main_2d(paths: Mapping[str, Path], device: torch.device, seeds: Sequence[int], smoke: bool = False, amp: bool = False) -> pd.DataFrame:
    cfg = Config2D(num_train=32 if smoke else 256, num_val=32 if smoke else 128, epochs=1 if smoke else 32, amp=amp)
    models = ["coord", "learned_warp", "so2_stn", "ga_rotor"]
    rows = []
    weights = LossWeights()
    for model_name in models:
        for seed in seeds:
            print(f"[main_2d] model={model_name} seed={seed}")
            save_path = paths["models"] / f"main2d_{model_name}_seed{seed}.pt" if seed == seeds[0] else None
            metrics, model, rep_batch = train_one_2d(model_name, cfg, seed, device, weights, save_path)
            rows.append({"experiment": "main_2d", "model": model_name, "seed": seed, **metrics})
            if model is not None and rep_batch is not None and model_name == "ga_rotor":
                save_2d_qualitative(model, rep_batch, device, paths["figures"] / "fig_2d_qualitative.pdf")
    df = pd.DataFrame(rows)
    df.to_csv(paths["csv"] / "main_2d_per_seed.csv", index=False)
    write_summary_table(df, models, paths["tables"] / "table_main_2d_20seed.tex")
    paired_tests_table(df, "coord", ["learned_warp", "so2_stn", "ga_rotor"], paths["tables"] / "table_paired_tests.tex")
    save_metric_bar_figure(df, models, paths["figures"] / "fig_main_2d_20seed.pdf")
    # Per-seed appendix table, compact CSV-like LaTeX.
    df.round(5).to_latex(paths["tables"] / "table_per_seed_main_2d.tex", index=False, escape=False)
    return df


def run_procedural(paths: Mapping[str, Path], device: torch.device, seeds: Sequence[int], smoke: bool = False, amp: bool = False) -> pd.DataFrame:
    cfg = Config2D(num_train=32 if smoke else 512, num_val=32 if smoke else 256, epochs=1 if smoke else 32, shape_kind="procedural", amp=amp)
    models = ["coord", "learned_warp", "so2_stn", "ga_rotor"]
    rows = []
    for model_name in models:
        for seed in seeds:
            print(f"[procedural] model={model_name} seed={seed}")
            metrics, _, _ = train_one_2d(model_name, cfg, seed + 20_000, device, LossWeights())
            rows.append({"experiment": "procedural", "model": model_name, "seed": seed, **metrics})
    df = pd.DataFrame(rows)
    df.to_csv(paths["csv"] / "procedural_per_seed.csv", index=False)
    return df


def run_angle(paths: Mapping[str, Path], device: torch.device, seeds: Sequence[int], smoke: bool = False, amp: bool = False) -> pd.DataFrame:
    rows = []
    models = ["coord", "learned_warp", "so2_stn", "ga_rotor"]
    settings = [("sparse45_to_uniform", "sparse45", "uniform"), ("range60_to_uniform", "range60", "uniform")]
    for setting_name, train_mode, val_mode in settings:
        cfg = Config2D(num_train=32 if smoke else 512, num_val=32 if smoke else 256, epochs=1 if smoke else 32, train_angle_mode=train_mode, val_angle_mode=val_mode, amp=amp)
        for model_name in models:
            for seed in seeds:
                print(f"[angle] setting={setting_name} model={model_name} seed={seed}")
                metrics, _, _ = train_one_2d(model_name, cfg, seed + 30_000, device, LossWeights())
                rows.append({"experiment": "angle", "angle_setting": setting_name, "model": model_name, "seed": seed, **metrics})
    df = pd.DataFrame(rows)
    df.to_csv(paths["csv"] / "angle_generalisation_per_seed.csv", index=False)
    return df



def make_generalisation_table(angle_df: Optional[pd.DataFrame], proc_df: Optional[pd.DataFrame], paths: Mapping[str, Path]) -> None:
    """Write the angle/generalisation and procedural-shape table."""
    row_end = r" \\"
    lines = [r"\begin{tabular}{llcc}", r"\toprule", r"Setting & Model & View Dice $\uparrow$ & Composition MSE $\downarrow$" + row_end, r"\midrule"]
    blocks: List[Tuple[str, pd.DataFrame]] = []
    if angle_df is not None and not angle_df.empty:
        for setting in sorted(angle_df["angle_setting"].unique()):
            blocks.append((setting.replace("_", " "), angle_df[angle_df["angle_setting"] == setting]))
    if proc_df is not None and not proc_df.empty:
        blocks.append(("procedural shapes", proc_df))
    for setting, subdf in blocks:
        models = [m for m in ["coord", "learned_warp", "so2_stn", "ga_rotor"] if m in set(subdf["model"])]
        mean_view = {m: subdf[subdf["model"] == m]["view_hard_dice"].mean() for m in models}
        mean_comp = {m: subdf[subdf["model"] == m]["composition_mse"].mean() for m in models}
        best_view = max(mean_view, key=mean_view.get)
        best_comp = min(mean_comp, key=mean_comp.get)
        for i, m in enumerate(models):
            vd_m, vd_s = mean_std(subdf[subdf["model"] == m]["view_hard_dice"].values)
            cm_m, cm_s = mean_std(subdf[subdf["model"] == m]["composition_mse"].values)
            vd = format_value(vd_m, vd_s, bold=m == best_view)
            cm = format_value(cm_m, cm_s, bold=m == best_comp)
            lines.append(f"{setting if i == 0 else ''} & {MODEL_LABELS.get(m,m)} & {vd} & {cm}" + row_end)
        lines.append(r"\midrule")
    if lines[-1] == r"\midrule":
        lines.pop()
    lines += [r"\bottomrule", r"\end{tabular}"]
    (paths["tables"] / "table_generalisation.tex").write_text("\n".join(lines), encoding="utf-8")


def run_scaling(paths: Mapping[str, Path], device: torch.device, seeds: Sequence[int], smoke: bool = False, amp: bool = False) -> pd.DataFrame:
    Ns = [32] if smoke else [256, 1024, 4096]
    caps = [(16, 8)] if smoke else [(16, 8), (32, 16), (64, 32)]
    models = ["coord", "ga_rotor"]  # scaling focuses on the main contrast to keep runtime tractable.
    rows = []
    for N in Ns:
        for z_dim, C in caps:
            cfg = Config2D(num_train=N, num_val=min(max(128, N // 4), 1024), epochs=1 if smoke else 32, z_dim=z_dim, base_channels=C, amp=amp)
            for model_name in models:
                for seed in seeds:
                    print(f"[scaling] N={N} z={z_dim} C={C} model={model_name} seed={seed}")
                    metrics, _, _ = train_one_2d(model_name, cfg, seed + 40_000 + N + z_dim, device, LossWeights())
                    rows.append({"experiment": "scaling", "num_train": N, "z_dim": z_dim, "base_channels": C, "model": model_name, "seed": seed, **metrics})
    df = pd.DataFrame(rows)
    df.to_csv(paths["csv"] / "scaling_per_seed.csv", index=False)
    row_end = r" \\"
    lines = [r"\begin{tabular}{lccc}", r"\toprule", r"Data $N$ / capacity $(d,C)$ & View Dice effect & Composition effect & Interpretation" + row_end, r"\midrule"]
    for (N, z, C), sub in df.groupby(["num_train", "z_dim", "base_channels"]):
        coord = sub[sub["model"] == "coord"].set_index("seed")
        rotor = sub[sub["model"] == "ga_rotor"].set_index("seed")
        common = sorted(set(coord.index) & set(rotor.index))
        vd_eff = (rotor.loc[common, "view_hard_dice"].values - coord.loc[common, "view_hard_dice"].values) / np.maximum(1e-8, coord.loc[common, "view_hard_dice"].values) * 100
        comp_eff = (coord.loc[common, "composition_mse"].values - rotor.loc[common, "composition_mse"].values) / np.maximum(1e-8, coord.loc[common, "composition_mse"].values) * 100
        vd_m, vd_s = mean_std(vd_eff)
        cm_m, cm_s = mean_std(comp_eff)
        interp = "persistent" if vd_m > 1.0 and cm_m > 10.0 else "shrinking/comparable"
        lines.append(f"{N} / ({z},{C}) & {vd_m:+.1f}$\\pm${vd_s:.1f}\\% & {cm_m:+.1f}$\\pm${cm_s:.1f}\\% & {interp}" + row_end)
    lines += [r"\bottomrule", r"\end{tabular}"]
    (paths["tables"] / "table_scaling.tex").write_text("\n".join(lines), encoding="utf-8")
    df.round(5).to_latex(paths["tables"] / "table_scaling_full.tex", index=False, escape=False)
    save_scaling_figure(df, paths["figures"] / "fig_scaling.pdf")
    return df

def save_scaling_figure(df: pd.DataFrame, outpath: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.4))
    for (z, C), subcap in df.groupby(["z_dim", "base_channels"]):
        xs, vd, comp = [], [], []
        for N, sub in sorted(subcap.groupby("num_train")):
            coord = sub[sub["model"] == "coord"].set_index("seed")
            rotor = sub[sub["model"] == "ga_rotor"].set_index("seed")
            common = sorted(set(coord.index) & set(rotor.index))
            if not common:
                continue
            xs.append(N)
            vd.append(float(np.mean(rotor.loc[common, "view_hard_dice"].values - coord.loc[common, "view_hard_dice"].values)))
            comp.append(float(np.mean(coord.loc[common, "composition_mse"].values - rotor.loc[common, "composition_mse"].values)))
        axes[0].plot(xs, vd, marker="o", label=f"d={z}, C={C}")
        axes[1].plot(xs, comp, marker="o", label=f"d={z}, C={C}")
    for ax, title, ylabel in [(axes[0], "View Dice gap", "rotor - coord"), (axes[1], "Composition MSE gap", "coord - rotor")]:
        ax.set_xscale("log", base=2)
        ax.set_xlabel("training pairs")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def run_loss_ablation(paths: Mapping[str, Path], device: torch.device, seeds: Sequence[int], smoke: bool = False, amp: bool = False) -> pd.DataFrame:
    cfg = Config2D(num_train=32 if smoke else 256, num_val=32 if smoke else 128, epochs=1 if smoke else 32, amp=amp)
    models = ["coord", "ga_rotor"]
    rows = []
    for variant, weights in LOSS_VARIANTS.items():
        for model_name in models:
            for seed in seeds:
                print(f"[loss] variant={variant} model={model_name} seed={seed}")
                metrics, _, _ = train_one_2d(model_name, cfg, seed + 50_000, device, weights)
                rows.append({"experiment": "loss_ablation", "loss_variant": variant, "model": model_name, "seed": seed, **metrics})
    df = pd.DataFrame(rows)
    df.to_csv(paths["csv"] / "loss_ablation_per_seed.csv", index=False)
    make_loss_table(df, paths["tables"] / "table_loss_ablation.tex")
    save_loss_figure(df, paths["figures"] / "fig_loss_ablation.pdf")
    return df



def make_loss_table(df: pd.DataFrame, outpath: Path) -> None:
    row_end = r" \\"
    lines = [r"\begin{tabular}{llccc}", r"\toprule", r"Loss variant & Model & View Dice $\uparrow$ & Comp. MSE $\downarrow$ & Entropy $\downarrow$" + row_end, r"\midrule"]
    for variant in LOSS_VARIANTS.keys():
        sub = df[df["loss_variant"] == variant]
        best_vd = sub.groupby("model")["view_hard_dice"].mean().idxmax()
        best_cm = sub.groupby("model")["composition_mse"].mean().idxmin()
        best_en = sub.groupby("model")["entropy"].mean().idxmin()
        for i, model in enumerate(["coord", "ga_rotor"]):
            ss = sub[sub["model"] == model]
            vd_m, vd_s = mean_std(ss["view_hard_dice"].values)
            cm_m, cm_s = mean_std(ss["composition_mse"].values)
            en_m, en_s = mean_std(ss["entropy"].values)
            lines.append(f"{variant if i==0 else ''} & {MODEL_LABELS[model]} & {format_value(vd_m, vd_s, model==best_vd)} & {format_value(cm_m, cm_s, model==best_cm)} & {format_value(en_m, en_s, model==best_en)}" + row_end)
        lines.append(r"\midrule")
    if lines[-1] == r"\midrule":
        lines.pop()
    lines += [r"\bottomrule", r"\end{tabular}"]
    outpath.write_text("\n".join(lines), encoding="utf-8")

def save_loss_figure(df: pd.DataFrame, outpath: Path) -> None:
    variants = list(LOSS_VARIANTS.keys())
    fig, ax = plt.subplots(figsize=(8.5, 3.5))
    x = np.arange(len(variants))
    width = 0.35
    for j, model in enumerate(["coord", "ga_rotor"]):
        means = [df[(df["loss_variant"] == v) & (df["model"] == model)]["view_hard_dice"].mean() for v in variants]
        ax.bar(x + (j - 0.5) * width, means, width, label=MODEL_LABELS[model])
    ax.set_xticks(x); ax.set_xticklabels(variants, rotation=35, ha="right")
    ax.set_ylabel("View Dice")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def run_three_d(paths: Mapping[str, Path], device: torch.device, seeds: Sequence[int], smoke: bool = False, amp: bool = False) -> pd.DataFrame:
    cfg = Config3D(size=16 if smoke else 32, num_train=16 if smoke else 512, num_val=8 if smoke else 128, batch_size=4 if smoke else 8, epochs=1 if smoke else 24, z_dim=16 if smoke else 32, base_channels=4 if smoke else 8, amp=amp)
    models = ["coord3d", "learned_warp3d", "so3_matrix", "rotor3d"]
    rows = []
    for model_name in models:
        for seed in seeds:
            print(f"[3d] model={model_name} seed={seed}")
            save_path = paths["models"] / f"3d_{model_name}_seed{seed}.pt" if seed == seeds[0] else None
            metrics, model, rep_batch = train_one_3d(model_name, cfg, seed + 60_000, device, LossWeights(), save_path)
            rows.append({"experiment": "three_d", "model": model_name, "seed": seed, **metrics})
            if model is not None and rep_batch is not None and model_name == "rotor3d":
                save_3d_qualitative(model, rep_batch, device, paths["figures"] / "fig_3d_qualitative.pdf")
    df = pd.DataFrame(rows)
    df.to_csv(paths["csv"] / "three_d_per_seed.csv", index=False)
    write_summary_table(df, models, paths["tables"] / "table_3d.tex")
    df.round(5).to_latex(paths["tables"] / "table_3d_per_seed.tex", index=False, escape=False)
    save_metric_bar_figure(df, models, paths["figures"] / "fig_3d_metrics.pdf")
    return df


def zip_results(outdir: Path) -> Path:
    zip_path = outdir / "tmlr_revision_results_upload.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in outdir.rglob("*"):
            if p == zip_path or p.is_dir():
                continue
            if "models" in p.parts:
                # keep zip small
                continue
            zf.write(p, p.relative_to(outdir))
    return zip_path


# -----------------------------------------------------------------------------
# CLI
# -----------------------------------------------------------------------------


def build_argparser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Run TMLR revision experiments and write paper-ready outputs.")
    p.add_argument("--outdir", type=str, default="tmlr_revision_results")
    p.add_argument("--experiments", type=str, nargs="+", default=["all"], choices=["main_2d", "scaling", "angle", "procedural", "loss_ablation", "three_d", "all"])
    p.add_argument("--seeds", type=str, default="0-9", help="Seed list, e.g. '0-9' or '0,1,2'.")
    p.add_argument("--scaling-seeds", type=str, default=None, help="Optional seed subset for scaling; defaults to --seeds.")
    p.add_argument("--three-d-seeds", type=str, default=None, help="Optional seed subset for 3D; defaults to --seeds.")
    p.add_argument("--device", type=str, default="cuda", choices=["cuda", "cpu"])
    p.add_argument("--amp", action="store_true", help="Use CUDA mixed precision where safe.")
    p.add_argument("--smoke", action="store_true", help="Very small one-epoch run to verify the pipeline.")
    return p


def main() -> None:
    args, _unknown = build_argparser().parse_known_args()
    outdir = Path(args.outdir)
    paths = ensure_dirs(outdir)
    seeds = parse_seed_string(args.seeds)
    if args.smoke:
        seeds = seeds[:1]
    scaling_seeds = parse_seed_string(args.scaling_seeds) if args.scaling_seeds else seeds
    three_d_seeds = parse_seed_string(args.three_d_seeds) if args.three_d_seeds else seeds
    if args.smoke:
        scaling_seeds = scaling_seeds[:1]
        three_d_seeds = three_d_seeds[:1]
    device = torch.device("cuda" if args.device == "cuda" and torch.cuda.is_available() else "cpu")
    # Keep CPU smoke tests responsive, Colab/A100 runs use CUDA kernels.
    torch.set_num_threads(1 if device.type == "cpu" else max(1, min(4, os.cpu_count() or 1)))
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass
    print(f"Device: {device}; seeds={seeds}; smoke={args.smoke}")
    experiments = args.experiments
    if "all" in experiments:
        experiments = ["main_2d", "scaling", "angle", "procedural", "loss_ablation", "three_d"]
    write_json({"args": vars(args), "device": str(device), "seeds": seeds, "start_time": time.asctime()}, paths["logs"] / "run_config.json")

    angle_df = None
    proc_df = None
    all_dfs: List[pd.DataFrame] = []
    if "main_2d" in experiments:
        all_dfs.append(run_main_2d(paths, device, seeds, smoke=args.smoke, amp=args.amp))
    if "scaling" in experiments:
        all_dfs.append(run_scaling(paths, device, scaling_seeds, smoke=args.smoke, amp=args.amp))
    if "angle" in experiments:
        angle_df = run_angle(paths, device, seeds, smoke=args.smoke, amp=args.amp)
        all_dfs.append(angle_df)
    if "procedural" in experiments:
        proc_df = run_procedural(paths, device, seeds, smoke=args.smoke, amp=args.amp)
        all_dfs.append(proc_df)
    if angle_df is not None or proc_df is not None:
        make_generalisation_table(angle_df, proc_df, paths)
    if "loss_ablation" in experiments:
        all_dfs.append(run_loss_ablation(paths, device, seeds, smoke=args.smoke, amp=args.amp))
    if "three_d" in experiments:
        all_dfs.append(run_three_d(paths, device, three_d_seeds, smoke=args.smoke, amp=args.amp))

    if all_dfs:
        pd.concat(all_dfs, ignore_index=True, sort=False).to_csv(paths["csv"] / "all_results_long.csv", index=False)
    zip_path = zip_results(outdir)
    write_json({"finished": time.asctime(), "zip_path": str(zip_path)}, paths["logs"] / "finished.json")
    print(f"Done: {zip_path}")


if __name__ == "__main__":
    main()


Device: cuda; seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]; smoke=False
[main_2d] model=coord seed=0


/tmp/ipykernel_614/1793186626.py:141: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(np.uint8(arr * 255.0), mode="L")


[main_2d] model=coord seed=1


/tmp/ipykernel_614/1793186626.py:141: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(np.uint8(arr * 255.0), mode="L")


[main_2d] model=coord seed=2
[main_2d] model=coord seed=3
[main_2d] model=coord seed=4
[main_2d] model=coord seed=5
[main_2d] model=coord seed=6
[main_2d] model=coord seed=7
[main_2d] model=coord seed=8
[main_2d] model=coord seed=9
[main_2d] model=learned_warp seed=0
[main_2d] model=learned_warp seed=1
[main_2d] model=learned_warp seed=2
[main_2d] model=learned_warp seed=3
[main_2d] model=learned_warp seed=4
[main_2d] model=learned_warp seed=5
[main_2d] model=learned_warp seed=6
[main_2d] model=learned_warp seed=7
[main_2d] model=learned_warp seed=8
[main_2d] model=learned_warp seed=9
[main_2d] model=so2_stn seed=0
[main_2d] model=so2_stn seed=1
[main_2d] model=so2_stn seed=2
[main_2d] model=so2_stn seed=3
[main_2d] model=so2_stn seed=4
[main_2d] model=so2_stn seed=5
[main_2d] model=so2_stn seed=6
[main_2d] model=so2_stn seed=7
[main_2d] model=so2_stn seed=8
[main_2d] model=so2_stn seed=9
[main_2d] model=ga_rotor seed=0
[main_2d] model=ga_rotor seed=1
[main_2d] model=ga_rotor seed=2
[m

/tmp/ipykernel_614/1793186626.py:141: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(np.uint8(arr * 255.0), mode="L")


[scaling] N=256 z=16 C=8 model=coord seed=1
[scaling] N=256 z=16 C=8 model=coord seed=2
[scaling] N=256 z=16 C=8 model=coord seed=3
[scaling] N=256 z=16 C=8 model=coord seed=4
[scaling] N=256 z=16 C=8 model=coord seed=5
[scaling] N=256 z=16 C=8 model=coord seed=6
[scaling] N=256 z=16 C=8 model=coord seed=7
[scaling] N=256 z=16 C=8 model=coord seed=8
[scaling] N=256 z=16 C=8 model=coord seed=9
[scaling] N=256 z=16 C=8 model=ga_rotor seed=0
[scaling] N=256 z=16 C=8 model=ga_rotor seed=1
[scaling] N=256 z=16 C=8 model=ga_rotor seed=2
[scaling] N=256 z=16 C=8 model=ga_rotor seed=3
[scaling] N=256 z=16 C=8 model=ga_rotor seed=4
[scaling] N=256 z=16 C=8 model=ga_rotor seed=5
[scaling] N=256 z=16 C=8 model=ga_rotor seed=6
[scaling] N=256 z=16 C=8 model=ga_rotor seed=7
[scaling] N=256 z=16 C=8 model=ga_rotor seed=8
[scaling] N=256 z=16 C=8 model=ga_rotor seed=9
[scaling] N=256 z=32 C=16 model=coord seed=0
[scaling] N=256 z=32 C=16 model=coord seed=1
[scaling] N=256 z=32 C=16 model=coord seed=2

/tmp/ipykernel_614/1793186626.py:141: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(np.uint8(arr * 255.0), mode="L")


[angle] setting=sparse45_to_uniform model=coord seed=1
[angle] setting=sparse45_to_uniform model=coord seed=2
[angle] setting=sparse45_to_uniform model=coord seed=3
[angle] setting=sparse45_to_uniform model=coord seed=4
[angle] setting=sparse45_to_uniform model=coord seed=5
[angle] setting=sparse45_to_uniform model=coord seed=6
[angle] setting=sparse45_to_uniform model=coord seed=7
[angle] setting=sparse45_to_uniform model=coord seed=8
[angle] setting=sparse45_to_uniform model=coord seed=9
[angle] setting=sparse45_to_uniform model=learned_warp seed=0
[angle] setting=sparse45_to_uniform model=learned_warp seed=1
[angle] setting=sparse45_to_uniform model=learned_warp seed=2
[angle] setting=sparse45_to_uniform model=learned_warp seed=3
[angle] setting=sparse45_to_uniform model=learned_warp seed=4
[angle] setting=sparse45_to_uniform model=learned_warp seed=5
[angle] setting=sparse45_to_uniform model=learned_warp seed=6
[angle] setting=sparse45_to_uniform model=learned_warp seed=7
[angle] s